# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN

### Thành viên trong nhóm:
- Võ Lê Ngọc Thịnh - 24521710
- Vũ Minh Phương - 24521421
- Nguyễn Hồng Phúc - 24521390
- Nguyễn Duy Khang - 24520755

# Demo Chương 18: Context-Free Grammars (CFG) và Constituency Parsing

Notebook này minh họa các nội dung chính của chương về **Context-Free Grammars (CFG)** và **Constituency Parsing** thông qua một demo nhỏ bằng tiếng Việt.

Nội dung bao gồm:

1. Xây dựng CFG thủ công.
2. Áp dụng CFG lên một mini dataset tiếng Việt tự tạo.
3. Minh họa hiện tượng nhập nhằng cấu trúc (*structural ambiguity*).
4. Chuyển grammar về Chomsky Normal Form (CNF).
5. Cài đặt thuật toán CKY (*Cocke-Kasami-Younger*) từ đầu.
6. Mở rộng: PCFG, treebank, đánh giá parser và giới hạn của CFG thủ công.

**Dataset sử dụng:** mini dataset tiếng Việt đời thường.

## 1. Ý tưởng chính

Một **Context-Free Grammar (CFG)** gồm 4 thành phần:

- Tập ký hiệu không kết thúc (*non-terminal symbols*), ví dụ: `S`, `NP`, `VP`, `PP`.
- Tập ký hiệu kết thúc (*terminal symbols*), tức là các từ trong câu.
- Tập luật sinh (*production rules*), ví dụ: `S -> NP VP`.
- Ký hiệu bắt đầu, thường là `S`.

Trong constituency parsing, mục tiêu là tìm cây cú pháp biểu diễn cách các từ trong câu kết hợp thành các cụm lớn hơn.

Ví dụ trực giác:

```text
S
├── NP
│   └── tôi
└── VP
    ├── thấy
    └── NP
        └── sinh_viên
```

## 2. Chuẩn bị môi trường

In [1]:
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Tuple, Dict, Set, Optional

try:
    from nltk import Tree
    NLTK_AVAILABLE = True
except Exception:
    NLTK_AVAILABLE = False

print("NLTK available:", NLTK_AVAILABLE)

NLTK available: False


## 3. Mini dataset tiếng Việt

Ta dùng các câu tiếng Việt đơn giản thuộc miền đời thường. Để dễ xử lý, các từ ghép như `sinh viên`, `thư viện`, `kính viễn vọng` được viết bằng dấu gạch dưới.

Ví dụ:

- `tôi thấy sinh_viên`
- `sinh_viên đọc sách trong thư_viện`
- `tôi thấy người_đàn_ông với kính_viễn_vọng`

Cách viết này giúp demo tập trung vào parsing thay vì bài toán tách từ tiếng Việt.

In [2]:
mini_dataset = [
    "tôi thấy sinh_viên",
    "sinh_viên đọc sách",
    "sinh_viên đọc sách trong thư_viện",
    "cô_giáo thấy sinh_viên trong thư_viện",
    "tôi thấy người_đàn_ông với kính_viễn_vọng",
    "tôi thấy con_mèo trên ban_công",
]

for i, sent in enumerate(mini_dataset, start=1):
    print(f"{i}. {sent}")

1. tôi thấy sinh_viên
2. sinh_viên đọc sách
3. sinh_viên đọc sách trong thư_viện
4. cô_giáo thấy sinh_viên trong thư_viện
5. tôi thấy người_đàn_ông với kính_viễn_vọng
6. tôi thấy con_mèo trên ban_công


## 4. Xây dựng CFG thủ công

Grammar dưới đây mô tả một phần nhỏ của tiếng Việt:

- `S -> NP VP`: một câu gồm cụm danh từ và cụm động từ.
- `VP -> V NP`: động từ đi với tân ngữ.
- `VP -> VP PP`: cụm giới từ bổ nghĩa cho cụm động từ.
- `NP -> NP PP`: cụm giới từ bổ nghĩa cho cụm danh từ.
- `PP -> P NP`: cụm giới từ gồm giới từ và cụm danh từ.

Hai luật `VP -> VP PP` và `NP -> NP PP` là nguồn quan trọng tạo ra nhập nhằng cấu trúc.

In [3]:
# Grammar dạng tổng quát, chưa cần CNF
CFG_RULES = {
    "S":  [("NP", "VP")],
    "VP": [("V", "NP"), ("VP", "PP"), ("V",)],
    "NP": [("N",), ("Pron",), ("Det", "N"), ("NP", "PP")],
    "PP": [("P", "NP")],
    "Pron": [("tôi",)],
    "Det": [("con",)],
    "N": [
        ("sinh_viên",),
        ("sách",),
        ("thư_viện",),
        ("cô_giáo",),
        ("người_đàn_ông",),
        ("kính_viễn_vọng",),
        ("mèo",),
        ("ban_công",),
    ],
    "V": [("thấy",), ("đọc",)],
    "P": [("trong",), ("với",), ("trên",)],
}

for lhs, rhss in CFG_RULES.items():
    rhs_text = " | ".join(" ".join(rhs) for rhs in rhss)
    print(f"{lhs} -> {rhs_text}")

S -> NP VP
VP -> V NP | VP PP | V
NP -> N | Pron | Det N | NP PP
PP -> P NP
Pron -> tôi
Det -> con
N -> sinh_viên | sách | thư_viện | cô_giáo | người_đàn_ông | kính_viễn_vọng | mèo | ban_công
V -> thấy | đọc
P -> trong | với | trên


## 5. Parser đệ quy đơn giản cho CFG

Trước khi dùng CKY, ta xây dựng một parser nhỏ theo kiểu đệ quy để dễ nhìn thấy cách CFG tạo cây.

Hàm `parse_symbol(symbol, tokens, i, j)` sẽ tìm tất cả cây có gốc là `symbol` sinh ra đoạn `tokens[i:j]`.

In [4]:
@dataclass(frozen=True)
class ParseNode:
    label: str
    children: Tuple

    def pretty(self, indent: int = 0) -> str:
        pad = "  " * indent
        if len(self.children) == 1 and isinstance(self.children[0], str):
            return f"{pad}({self.label} {self.children[0]})"
        lines = [f"{pad}({self.label}"]
        for child in self.children:
            if isinstance(child, ParseNode):
                lines.append(child.pretty(indent + 1))
            else:
                lines.append("  " * (indent + 1) + str(child))
        lines[-1] += ")"
        return "\n".join(lines)

    def to_nltk(self):
        return Tree(self.label, [child.to_nltk() if isinstance(child, ParseNode) else child for child in self.children])


def parse_cfg(tokens: List[str], start_symbol: str = "S") -> List[ParseNode]:
    memo = {}

    def parse_symbol(symbol: str, i: int, j: int) -> List[ParseNode]:
        key = (symbol, i, j)
        if key in memo:
            return memo[key]

        results = []
        for rhs in CFG_RULES.get(symbol, []):
            # Trường hợp lexical: A -> word
            if len(rhs) == 1 and rhs[0] not in CFG_RULES:
                if j == i + 1 and tokens[i] == rhs[0]:
                    results.append(ParseNode(symbol, (rhs[0],)))

            # Trường hợp unary non-terminal: A -> B
            elif len(rhs) == 1:
                B = rhs[0]
                for child in parse_symbol(B, i, j):
                    results.append(ParseNode(symbol, (child,)))

            # Trường hợp binary: A -> B C
            elif len(rhs) == 2:
                B, C = rhs
                for k in range(i + 1, j):
                    left_trees = parse_symbol(B, i, k)
                    right_trees = parse_symbol(C, k, j)
                    for left in left_trees:
                        for right in right_trees:
                            results.append(ParseNode(symbol, (left, right)))

        memo[key] = results
        return results

    return parse_symbol(start_symbol, 0, len(tokens))

## 6. Áp dụng CFG lên mini dataset

Ta chạy parser trên toàn bộ dataset và đếm số cây parse tìm được cho từng câu.

Nếu một câu có nhiều hơn một cây parse, câu đó có nhập nhằng cấu trúc theo grammar hiện tại.

In [5]:
for sent in mini_dataset:
    tokens = sent.split()
    parses = parse_cfg(tokens)
    print("Câu:", sent)
    print("Số cây parse:", len(parses))
    if parses:
        print(parses[0].pretty())
    print("-" * 70)

Câu: tôi thấy sinh_viên
Số cây parse: 1
(S
  (NP
    (Pron tôi))
  (VP
    (V thấy)
    (NP
      (N sinh_viên))))
----------------------------------------------------------------------
Câu: sinh_viên đọc sách
Số cây parse: 1
(S
  (NP
    (N sinh_viên))
  (VP
    (V đọc)
    (NP
      (N sách))))
----------------------------------------------------------------------
Câu: sinh_viên đọc sách trong thư_viện
Số cây parse: 2
(S
  (NP
    (N sinh_viên))
  (VP
    (V đọc)
    (NP
      (NP
        (N sách))
      (PP
        (P trong)
        (NP
          (N thư_viện))))))
----------------------------------------------------------------------
Câu: cô_giáo thấy sinh_viên trong thư_viện
Số cây parse: 2
(S
  (NP
    (N cô_giáo))
  (VP
    (V thấy)
    (NP
      (NP
        (N sinh_viên))
      (PP
        (P trong)
        (NP
          (N thư_viện))))))
----------------------------------------------------------------------
Câu: tôi thấy người_đàn_ông với kính_viễn_vọng
Số cây parse: 2
(S
  (NP

## 7. Hiện tượng nhập nhằng cấu trúc

Xét câu:

```text
tôi thấy người_đàn_ông với kính_viễn_vọng
```

Câu này có ít nhất hai cách hiểu:

1. Tôi dùng kính viễn vọng để thấy người đàn ông.  
   Khi đó cụm `với kính_viễn_vọng` bổ nghĩa cho hành động `thấy`, tức là gắn vào `VP`.

2. Tôi thấy người đàn ông đang có kính viễn vọng.  
   Khi đó cụm `với kính_viễn_vọng` bổ nghĩa cho `người_đàn_ông`, tức là gắn vào `NP`.

Đây là ví dụ kinh điển của **PP attachment ambiguity**.

In [6]:
ambiguous_sentence = "tôi thấy người_đàn_ông với kính_viễn_vọng"
tokens = ambiguous_sentence.split()
ambiguous_parses = parse_cfg(tokens)

print("Câu nhập nhằng:", ambiguous_sentence)
print("Số cây parse tìm được:", len(ambiguous_parses))
print()

for idx, tree in enumerate(ambiguous_parses, start=1):
    print(f"Cây parse {idx}:")
    print(tree.pretty())
    print("-" * 70)

Câu nhập nhằng: tôi thấy người_đàn_ông với kính_viễn_vọng
Số cây parse tìm được: 2

Cây parse 1:
(S
  (NP
    (Pron tôi))
  (VP
    (V thấy)
    (NP
      (NP
        (N người_đàn_ông))
      (PP
        (P với)
        (NP
          (N kính_viễn_vọng))))))
----------------------------------------------------------------------
Cây parse 2:
(S
  (NP
    (Pron tôi))
  (VP
    (VP
      (V thấy)
      (NP
        (N người_đàn_ông)))
    (PP
      (P với)
      (NP
        (N kính_viễn_vọng)))))
----------------------------------------------------------------------


In [7]:
# Nếu có NLTK, hiển thị một cây theo dạng bracketed tree đẹp hơn.
if NLTK_AVAILABLE and ambiguous_parses:
    ambiguous_parses[0].to_nltk()
else:
    print("Không có NLTK hoặc không có cây parse để hiển thị.")

Không có NLTK hoặc không có cây parse để hiển thị.


## 8. Vì sao cần Chomsky Normal Form?

Thuật toán CKY hoạt động với grammar ở **Chomsky Normal Form (CNF)**.

Một grammar ở CNF có các luật dạng:

```text
A -> B C
A -> word
```

Trong đó:

- `A`, `B`, `C` là non-terminal.
- `word` là terminal.

CKY dựa trên ý tưởng quy hoạch động: nếu một đoạn con của câu có thể tạo thành `B`, và đoạn con kế tiếp có thể tạo thành `C`, thì toàn bộ đoạn có thể tạo thành `A` nếu có luật `A -> B C`.

Ở phần tiếp theo, ta viết một grammar CNF nhỏ tương đương với phần cần dùng của grammar trên.

In [8]:
# Grammar CNF cho CKY
# Dạng lưu trữ:
# - lexical_rules[word] = {A | A -> word}
# - binary_rules[(B, C)] = {A | A -> B C}

cnf_productions = [
    # Binary rules
    ("S",  ("NP", "VP")),
    ("VP", ("V", "NP")),
    ("VP", ("VP", "PP")),
    ("NP", ("Det", "N")),
    ("NP", ("NP", "PP")),
    ("PP", ("P", "NP")),

    # Lexical rules
    ("NP", ("tôi",)),
    ("NP", ("sinh_viên",)),
    ("NP", ("cô_giáo",)),
    ("NP", ("sách",)),
    ("NP", ("thư_viện",)),
    ("NP", ("người_đàn_ông",)),
    ("NP", ("kính_viễn_vọng",)),
    ("N",  ("sinh_viên",)),
    ("N",  ("sách",)),
    ("N",  ("thư_viện",)),
    ("N",  ("người_đàn_ông",)),
    ("N",  ("kính_viễn_vọng",)),
    ("N",  ("mèo",)),
    ("N",  ("ban_công",)),
    ("Det", ("con",)),
    ("V",  ("thấy",)),
    ("V",  ("đọc",)),
    ("P",  ("trong",)),
    ("P",  ("với",)),
    ("P",  ("trên",)),
]

lexical_rules = defaultdict(set)
binary_rules = defaultdict(set)

for lhs, rhs in cnf_productions:
    if len(rhs) == 1:
        lexical_rules[rhs[0]].add(lhs)
    elif len(rhs) == 2:
        binary_rules[rhs].add(lhs)
    else:
        raise ValueError("CNF chỉ cho phép RHS có độ dài 1 hoặc 2")

print("Lexical rules:")
for word, lhs_set in lexical_rules.items():
    print(f"  {word:20s} -> {sorted(lhs_set)}")

print("\nBinary rules:")
for rhs, lhs_set in binary_rules.items():
    print(f"  {rhs} -> {sorted(lhs_set)}")

Lexical rules:
  tôi                  -> ['NP']
  sinh_viên            -> ['N', 'NP']
  cô_giáo              -> ['NP']
  sách                 -> ['N', 'NP']
  thư_viện             -> ['N', 'NP']
  người_đàn_ông        -> ['N', 'NP']
  kính_viễn_vọng       -> ['N', 'NP']
  mèo                  -> ['N']
  ban_công             -> ['N']
  con                  -> ['Det']
  thấy                 -> ['V']
  đọc                  -> ['V']
  trong                -> ['P']
  với                  -> ['P']
  trên                 -> ['P']

Binary rules:
  ('NP', 'VP') -> ['S']
  ('V', 'NP') -> ['VP']
  ('VP', 'PP') -> ['VP']
  ('Det', 'N') -> ['NP']
  ('NP', 'PP') -> ['NP']
  ('P', 'NP') -> ['PP']


## 9. Cài đặt CKY parser từ đầu

Bảng CKY là một bảng tam giác.

- Ô `(i, j)` chứa các non-terminal có thể sinh ra đoạn `tokens[i:j]`.
- Các ô đường chéo ứng với từng từ riêng lẻ.
- Các ô dài hơn được tính bằng cách thử mọi điểm tách `k` giữa `i` và `j`.

Công thức trực giác:

```text
Nếu B thuộc chart[i][k]
Và C thuộc chart[k][j]
Và có luật A -> B C
Thì thêm A vào chart[i][j]
```

In [9]:
def cky_parse(tokens: List[str], start_symbol: str = "S"):
    n = len(tokens)
    chart: List[List[Set[str]]] = [[set() for _ in range(n + 1)] for _ in range(n)]
    backpointers = defaultdict(list)

    # Bước 1: điền lexical rules
    for i, word in enumerate(tokens):
        for lhs in lexical_rules.get(word, set()):
            chart[i][i + 1].add(lhs)
            backpointers[(i, i + 1, lhs)].append((word,))

    # Bước 2: điền các span dài hơn
    for span in range(2, n + 1):
        for i in range(n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                for B in chart[i][k]:
                    for C in chart[k][j]:
                        for A in binary_rules.get((B, C), set()):
                            chart[i][j].add(A)
                            backpointers[(i, j, A)].append((k, B, C))

    accepted = start_symbol in chart[0][n] if n > 0 else False
    return chart, backpointers, accepted


def print_cky_chart(tokens: List[str], chart):
    n = len(tokens)
    print("Tokens:", tokens)
    print()
    for span in range(1, n + 1):
        print(f"Span length = {span}")
        for i in range(n - span + 1):
            j = i + span
            cell = ", ".join(sorted(chart[i][j])) if chart[i][j] else "∅"
            print(f"  chart[{i},{j}] ({' '.join(tokens[i:j])}) = {{{cell}}}")
        print()

## 10. Chạy CKY trên một câu đơn giản

Ta thử với câu:

```text
sinh_viên đọc sách trong thư_viện
```

Theo grammar, câu này có thể parse được vì:

- `sinh_viên` là `NP`.
- `đọc sách` là `VP`.
- `trong thư_viện` là `PP`.
- `đọc sách trong thư_viện` có thể là `VP -> VP PP`.
- Toàn câu là `S -> NP VP`.

In [10]:
sentence = "sinh_viên đọc sách trong thư_viện"
tokens = sentence.split()
chart, backpointers, accepted = cky_parse(tokens)

print("Câu:", sentence)
print("Parse thành công?", accepted)
print()
print_cky_chart(tokens, chart)

Câu: sinh_viên đọc sách trong thư_viện
Parse thành công? True

Tokens: ['sinh_viên', 'đọc', 'sách', 'trong', 'thư_viện']

Span length = 1
  chart[0,1] (sinh_viên) = {N, NP}
  chart[1,2] (đọc) = {V}
  chart[2,3] (sách) = {N, NP}
  chart[3,4] (trong) = {P}
  chart[4,5] (thư_viện) = {N, NP}

Span length = 2
  chart[0,2] (sinh_viên đọc) = {∅}
  chart[1,3] (đọc sách) = {VP}
  chart[2,4] (sách trong) = {∅}
  chart[3,5] (trong thư_viện) = {PP}

Span length = 3
  chart[0,3] (sinh_viên đọc sách) = {S}
  chart[1,4] (đọc sách trong) = {∅}
  chart[2,5] (sách trong thư_viện) = {NP}

Span length = 4
  chart[0,4] (sinh_viên đọc sách trong) = {∅}
  chart[1,5] (đọc sách trong thư_viện) = {VP}

Span length = 5
  chart[0,5] (sinh_viên đọc sách trong thư_viện) = {S}



## 11. Truy vết cây parse từ CKY

Ngoài việc trả lời câu có parse được hay không, ta có thể dùng backpointer để dựng lại cây parse.

In [11]:
def build_trees(i: int, j: int, symbol: str, backpointers) -> List[ParseNode]:
    trees = []
    for bp in backpointers.get((i, j, symbol), []):
        # lexical backpointer
        if len(bp) == 1 and isinstance(bp[0], str):
            trees.append(ParseNode(symbol, (bp[0],)))
        else:
            k, B, C = bp
            left_trees = build_trees(i, k, B, backpointers)
            right_trees = build_trees(k, j, C, backpointers)
            for left in left_trees:
                for right in right_trees:
                    trees.append(ParseNode(symbol, (left, right)))
    return trees

cky_trees = build_trees(0, len(tokens), "S", backpointers)
print("Số cây CKY dựng lại:", len(cky_trees))
for tree in cky_trees:
    print(tree.pretty())
    print("-" * 70)

Số cây CKY dựng lại: 2
(S
  (NP sinh_viên)
  (VP
    (V đọc)
    (NP
      (NP sách)
      (PP
        (P trong)
        (NP thư_viện)))))
----------------------------------------------------------------------
(S
  (NP sinh_viên)
  (VP
    (VP
      (V đọc)
      (NP sách))
    (PP
      (P trong)
      (NP thư_viện))))
----------------------------------------------------------------------


## 12. CKY và nhập nhằng cấu trúc

Bây giờ chạy CKY với câu nhập nhằng:

```text
tôi thấy người_đàn_ông với kính_viễn_vọng
```

Nếu grammar cho phép cả `VP -> VP PP` và `NP -> NP PP`, CKY có thể tìm nhiều cây parse cho cùng một câu.

In [12]:
sentence = "tôi thấy người_đàn_ông với kính_viễn_vọng"
tokens = sentence.split()
chart, backpointers, accepted = cky_parse(tokens)
cky_trees = build_trees(0, len(tokens), "S", backpointers)

print("Câu:", sentence)
print("Parse thành công?", accepted)
print("Số cây parse CKY:", len(cky_trees))
print()

for idx, tree in enumerate(cky_trees, start=1):
    print(f"Cây CKY {idx}:")
    print(tree.pretty())
    print("-" * 70)

Câu: tôi thấy người_đàn_ông với kính_viễn_vọng
Parse thành công? True
Số cây parse CKY: 2

Cây CKY 1:
(S
  (NP tôi)
  (VP
    (V thấy)
    (NP
      (NP người_đàn_ông)
      (PP
        (P với)
        (NP kính_viễn_vọng)))))
----------------------------------------------------------------------
Cây CKY 2:
(S
  (NP tôi)
  (VP
    (VP
      (V thấy)
      (NP người_đàn_ông))
    (PP
      (P với)
      (NP kính_viễn_vọng))))
----------------------------------------------------------------------


In [13]:
print_cky_chart(tokens, chart)

Tokens: ['tôi', 'thấy', 'người_đàn_ông', 'với', 'kính_viễn_vọng']

Span length = 1
  chart[0,1] (tôi) = {NP}
  chart[1,2] (thấy) = {V}
  chart[2,3] (người_đàn_ông) = {N, NP}
  chart[3,4] (với) = {P}
  chart[4,5] (kính_viễn_vọng) = {N, NP}

Span length = 2
  chart[0,2] (tôi thấy) = {∅}
  chart[1,3] (thấy người_đàn_ông) = {VP}
  chart[2,4] (người_đàn_ông với) = {∅}
  chart[3,5] (với kính_viễn_vọng) = {PP}

Span length = 3
  chart[0,3] (tôi thấy người_đàn_ông) = {S}
  chart[1,4] (thấy người_đàn_ông với) = {∅}
  chart[2,5] (người_đàn_ông với kính_viễn_vọng) = {NP}

Span length = 4
  chart[0,4] (tôi thấy người_đàn_ông với) = {∅}
  chart[1,5] (thấy người_đàn_ông với kính_viễn_vọng) = {VP}

Span length = 5
  chart[0,5] (tôi thấy người_đàn_ông với kính_viễn_vọng) = {S}



## 13. Kiểm thử trên mini dataset

Ta chạy CKY trên toàn bộ mini dataset để xem câu nào parse được, câu nào nhập nhằng.

In [14]:
summary = []

for sent in mini_dataset:
    tokens = sent.split()
    chart, backpointers, accepted = cky_parse(tokens)
    trees = build_trees(0, len(tokens), "S", backpointers) if accepted else []
    summary.append((sent, accepted, len(trees)))

print(f"{'Câu':55s} | {'Parse?':7s} | Số cây")
print("-" * 80)
for sent, accepted, n_trees in summary:
    print(f"{sent:55s} | {str(accepted):7s} | {n_trees}")

Câu                                                     | Parse?  | Số cây
--------------------------------------------------------------------------------
tôi thấy sinh_viên                                      | True    | 1
sinh_viên đọc sách                                      | True    | 1
sinh_viên đọc sách trong thư_viện                       | True    | 2
cô_giáo thấy sinh_viên trong thư_viện                   | True    | 2
tôi thấy người_đàn_ông với kính_viễn_vọng               | True    | 2
tôi thấy con_mèo trên ban_công                          | False   | 0


## 14. Mở rộng: nếu dùng PCFG thì sao?

CFG thường chỉ nói một câu có hợp lệ hay không, nhưng không nói cây nào hợp lý hơn khi có nhiều cây parse.

Một hướng mở rộng là **Probabilistic Context-Free Grammar (PCFG)**. PCFG gán xác suất cho từng luật, ví dụ:

```text
VP -> VP PP   [0.3]
NP -> NP PP   [0.2]
VP -> V NP    [0.7]
```

Khi đó, xác suất của một cây parse là tích xác suất của các luật được dùng trong cây. Parser có thể chọn cây có xác suất cao nhất.

Ý tưởng này hữu ích khi xử lý nhập nhằng, vì trong ngôn ngữ tự nhiên, không phải mọi cách phân tích đều có khả năng xuất hiện ngang nhau.

In [15]:
# Minh họa rất nhỏ về cách tính điểm PCFG cho hai kiểu gắn PP.
pcfg_rule_probs = {
    ("S",  ("NP", "VP")): 1.0,
    ("VP", ("V", "NP")): 0.7,
    ("VP", ("VP", "PP")): 0.3,
    ("NP", ("NP", "PP")): 0.2,
    ("PP", ("P", "NP")): 1.0,
}

print("Trong PCFG thực tế, các xác suất này thường được ước lượng từ treebank.")
print("Ở đây chỉ minh họa ý tưởng: cây có tích xác suất lớn hơn thường được ưu tiên.")

Trong PCFG thực tế, các xác suất này thường được ước lượng từ treebank.
Ở đây chỉ minh họa ý tưởng: cây có tích xác suất lớn hơn thường được ưu tiên.
